In [ ]:
# adaboost_regressor.ipynb

# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==========================================
# 2. LOAD DATASET
# ==========================================

df = pd.read_csv("../data/insurance.csv")

# ==========================================
# 3. DATA EXPLORATION
# ==========================================

print(df.head())

print(df.shape)

print(df.info())

print(df.isnull().sum())

# ==========================================
# 4. ENCODE CATEGORICAL COLUMNS
# ==========================================

df = pd.get_dummies(
    df,
    drop_first=True
)

# ==========================================
# 5. FEATURES AND TARGET
# ==========================================

X = df.drop("charges", axis=1)

y = df["charges"]

# Save feature columns
feature_columns = X.columns.tolist()

# ==========================================
# 6. TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# 7. MODEL TRAINING
# ==========================================

base_model = DecisionTreeRegressor(
    max_depth=4
)

model = AdaBoostRegressor(
    estimator=base_model,
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

# ==========================================
# 8. PREDICTIONS
# ==========================================

y_pred = model.predict(X_test)

# ==========================================
# 9. EVALUATION
# ==========================================

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("\nModel Evaluation")
print("-" * 30)

print(f"MAE  : {mae:.2f}")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R2   : {r2:.4f}")

# ==========================================
# 10. FEATURE IMPORTANCE
# ==========================================

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Features")

print(importance_df.head(10))

# Plot Feature Importance

plt.figure(figsize=(10, 6))

sns.barplot(
    x="Importance",
    y="Feature",
    data=importance_df.head(10)
)

plt.title("Top 10 Feature Importance")

plt.show()

# ==========================================
# 11. SAVE MODEL
# ==========================================

with open("../models/adaboost_model.pkl", "wb") as file:
    pickle.dump(model, file)

with open("../models/feature_columns.pkl", "wb") as file:
    pickle.dump(feature_columns, file)

print("\nModel Saved Successfully!")